# Week 8 · Tuesday: Vector databases, Chroma and Pinecone

Yesterday's credit table had about twenty columns per row, and every one was a number a model could read. Today each piece of text becomes 384 numbers, and we store and search them three ways: in an SQLite table, in Chroma on Colab's disk, and in a Pinecone index in the cloud. Today:

1. **Vector overview:** a vector places a thing, and cosine similarity says how alike two are
2. **Vector similarity search:** a model turns text into vectors, and a search compares a question with every stored one
3. **Storing and retrieving data in SQLite:** vectors in an ordinary table, and why it stops scaling
4. **Vector databases:** what they add over a table, and what they are used for
5. **Chroma DB setup:** installing Chroma and choosing a client
6. **A local vector database with Chroma:** a collection, a query, then create, read, update and delete
7. **Pinecone overview:** a managed vector database, its free plan, and keys as secrets
8. **Pinecone setup:** the client, the key from Colab Secrets, an index
9. **CRUD with vector data:** create, read, update and delete in Pinecone, then the three stores side by side

Sections 1 to 6 need no account. Sections 7 to 9 run against your own free Pinecone project and need its API key saved in Colab's Secrets panel as `PINECONE_API_KEY` (7.2 says how). Without a key, Runtime > Run all runs every Chroma part and stops at the first Pinecone cell, in 8.1.

Why the next three days exist, in one sentence: next week a language model answers questions about documents it has never seen, and this week builds the part that finds the right document.

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

**Studying on your own**

This notebook is written to be worked through alone as well as followed live. Allow about three and a quarter hours for all of it. The ends of sections 2, 4 and 6 are good places to stop for a break; the end of section 6 is also where the Chroma part finishes and the Pinecone part begins.

| Sections | What you work through | About |
|---|---|---|
| 1 and 2 | vectors and cosine similarity by hand, then text turned into vectors and searched | 40 min |
| 3 and 4.1 | vectors in SQLite, a timed search over 50,000 rows, what a vector database adds | 25 min |
| 4.2 | the use cases, starting from an app you use | 15 min |
| 5 and 6 | Chroma: the install, two clients, a collection and its CRUD calls | 35 min |
| 7 and 8 | Pinecone: the free plan, the key, an index | 30 min |
| 9 | CRUD in Pinecone, the clean-up and the three stores compared | 35 min |
| Recap | the Quick check | 10 min |

How to work a subsection:

- Read the text above the code cell and answer its **Predict first** question before you run anything. Write the answer down: a guess you have committed to is what makes the result stick.
- Run the cell. Compare what it prints with your prediction and with the bullets under it. The numbers quoted in the text come from our own run; yours should match or sit very close, and timings depend on the machine.
- Do the **Try changing** edit, see what moved, then put the value back. Later cells rely on the original values.
- A **Common mistake** bullet names an error people often make at that step.
- **Work it by hand first** marks an exercise to do on paper before reading on: cosine similarity in 1.2, and the use cases of an app you know in 4.2.
- A line such as "Check your answer" opens when you click it. Try first, then open it.
- Each section ends with **Before you move on**. If you cannot do one of its items, reread that subsection before going on.
- The Quick check at the end has its answers folded under it. Say each answer aloud before you open them: these are the questions an interviewer asks.
- If the runtime disconnects (Colab does this after a long idle spell), click the cell you were on and choose Runtime > Run before. That reruns the two installs (about a minute) and the cell in 5.2 empties the Chroma folder first, so every collection is made again cleanly. Every section from 3 on needs `kb_vectors` and `query_vector` from 2.3.
- If you stop partway through section 9, run the clean-up cell in 9.5 before you leave, so your free Pinecone project is empty and the next run starts clean.
- The [notes page for today](https://2494-gen-ai-cn.github.io/training/weeks/week-08/day-2-vector-databases-chroma-and-pinecone.html) covers the same ideas in prose with 10 of the 30 articles, goes further in places (Chroma as a server, pod-based against serverless indexes, API key permissions), and ends with a short check.

Run the next cell first. It holds the imports and the helpers we reuse all day, and it fixes the random seed so your numbers match the ones quoted in the text.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import os
import shutil
import sqlite3
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
IN_COLAB = "google.colab" in sys.modules        # True on Colab; the install lines in 5.1 and 8.1 run only there
pd.set_option("display.width", 120)


def plot_points(points, title=""):
    """Draw a dict of name -> [x, y] vector as labelled points on a 0-to-10 grid."""
    plt.figure(figsize=(5.5, 5.5))
    for name, (x, y) in points.items():
        plt.scatter(x, y, s=80)
        plt.annotate(name, (x, y), textcoords="offset points", xytext=(8, 4), fontsize=11)
    plt.xlim(0, 10.5)
    plt.ylim(0, 10.5)
    plt.xlabel("action")
    plt.ylabel("romance")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.show()


def plot_similarity(matrix, labels, title=""):
    """Draw a square table of similarities as a heat map, each cell's value written in it."""
    plt.figure(figsize=(8, 6.5))
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, square=True)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def chroma_hits(result):
    """A Chroma query result as a table: id, distance, then the display fields."""
    rows = []
    for id_, dist, meta in zip(result["ids"][0], result["distances"][0], result["metadatas"][0]):
        rows.append({"id": id_, "distance": round(dist, 2), "title": meta["title"],
                     "category": meta["category"], "product": meta["product"]})
    return pd.DataFrame(rows)


def pinecone_hits(result):
    """A Pinecone query result as a table: id, score, then the display fields."""
    return pd.DataFrame([{"id": m.id, "score": round(m.score, 2), "title": m.metadata["title"],
                          "category": m.metadata["category"], "product": m.metadata["product"]}
                         for m in result.matches])


def wait_for_count(index, expected, namespace=None):
    """Poll the index stats every 2 seconds, for up to 60 seconds, until it holds `expected` vectors."""
    for _ in range(30):
        stats = index.describe_index_stats()
        if namespace is None:
            count = stats.total_vector_count
        else:
            ns = stats.namespaces.get(namespace)
            count = ns.vector_count if ns else 0
        if count == expected:
            break
        time.sleep(2)
    print(f"{count} vectors in the index")


print(f"Python {sys.version.split()[0]}, NumPy {np.__version__}, pandas {pd.__version__}, "
      f"SQLite {sqlite3.sqlite_version}, on Colab: {IN_COLAB}")
print("Setup complete")

## 1. Vector overview

A **vector** is an ordered list of numbers. Two numbers place a point on a flat page, three place it in a room, and 384 place it in a space nobody can draw. The arithmetic is the same at any length, so we start with two.

### 1.1 A vector places a thing

Four films, each scored from 0 to 10 on two invented features, action and romance. A film's two scores, always in that order, are its vector. Drawn as points, the vectors show which films are alike.

An **embedding** is a vector that represents a piece of data (a word, a film, a paragraph), with its numbers chosen so that alike things sit close together. Turning data into such numbers is **vectorising**. It is what lets a program that only does arithmetic work with text or images.

**Predict first:** Star Run scores `[9, 1]`: lots of action, little romance. Which of the other three films will sit closest to it?

In [ ]:
films = {
    "Star Run":     np.array([9, 1]),   # [action, romance], invented scores
    "Moon Base":    np.array([8, 2]),
    "Love Letters": np.array([1, 9]),
    "Paris Days":   np.array([2, 8]),
}
for name, v in films.items():
    print(f"{name:13} {v}")
plot_points(films, title="Four films placed by two invented features")

- Moon Base sits closest to Star Run: both score high on action and low on romance. The two romance films sit together in the other corner.
- The position means the same thing in every vector: the first number is always action, the second always romance. That is what makes two vectors comparable.
- The limit: two numbers know nothing about a film's length or its language. A vector carries only what its numbers were built from.
- **Common mistake:** reading one number of a learned embedding as a feature, the way we read "action" here. These hand-made vectors have named numbers. The 384 numbers a model makes in 2.1 have no names, and only the position as a whole means something.

**In an interview:** a vector embedding is a list of numbers that represents a piece of data as a point, placed so that alike things sit close together. A program can then compare items by doing arithmetic on their vectors.

### 1.2 Cosine similarity

To say how alike two vectors are, we compare the directions they point in.

- The **dot product** multiplies the numbers in matching positions and adds the results.
- A vector's **length** (its norm) is the square root of the sum of its squared numbers: `[3, 4]` has length 5, `[6, 8]` has length 10.
- **Cosine similarity** is the dot product divided by the product of the two lengths. It measures how closely two vectors point the same way and ignores how long they are. It runs from -1 (opposite directions) through 0 (at right angles: nothing in common) to 1 (the same direction).

```
cosine(a, b) = (a · b) / (|a| × |b|)
```

Week 6's k-means used **Euclidean distance**, the straight-line distance between two points. Comparing it with cosine is Wednesday's; today every comparison is cosine.

**Work it by hand first.** On paper, work out two cosine similarities: `[3, 4]` with `[6, 8]`, and `[3, 4]` with `[4, -3]`. Write both numbers down before you open the answer or run the cell.

<details>
<summary>Check your answer</summary>

- `[3, 4]` and `[6, 8]`: the dot product is 3 × 6 + 4 × 8 = 50 and the lengths are 5 and 10, so 50 / (5 × 10) = **1.0**. The second vector is the first doubled: longer, same direction.
- `[3, 4]` and `[4, -3]`: the dot product is 3 × 4 + 4 × (-3) = 12 - 12 = 0, so the similarity is **0.0**. The two point at right angles.

Two usual wrong answers, and what causes each:

- Something below 1 for the first pair, such as 0.5, from expecting the longer vector to be "less like" the short one. Cosine ignores length; only the direction counts.
- A length of 7 for `[3, 4]`, from adding 3 + 4 instead of taking the square root of 9 + 16.

</details>

The cell below writes the formula line for line: `np.dot` is the top of the fraction and `np.linalg.norm` gives each length.

**Predict first:** `[6, 8]` is twice as long as `[3, 4]`. Will the cell print a cosine similarity for that pair below 1, exactly 1, or above 1?

In [ ]:
def cosine(a, b):
    """Cosine similarity: the dot product over the product of the two lengths."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

a, b, c = np.array([3, 4]), np.array([6, 8]), np.array([4, -3])   # Try changing it: c = np.array([-3, -4])
print(f"cosine(a, b) = {cosine(a, b):.2f}   cosine(a, c) = {cosine(a, c):.2f}")
print(f"Star Run against Moon Base {cosine(films['Star Run'], films['Moon Base']):.2f}, "
      f"against Love Letters {cosine(films['Star Run'], films['Love Letters']):.2f}")

- 1.00 and 0.00, the answers worked by hand.
- Star Run against Moon Base scores 0.99 and against Love Letters 0.22: the plot's two groups, now as numbers.
- When every vector has length 1, the bottom of the fraction is 1 × 1, so the dot product alone is the cosine similarity. Section 2 relies on that.
- **Common mistake:** expecting the longer vector to score higher, or lower. A vector and its double point the same way and say the same thing, so they score exactly 1.
- Try changing it: set `c = np.array([-3, -4])`, the first vector turned round, and rerun. `cosine(a, c)` prints -1.00, the bottom of the range. Put it back to `np.array([4, -3])` and rerun the cell.

**In an interview:** cosine similarity is the dot product of two vectors divided by the product of their lengths. It measures direction only, from -1 to 1, so a vector and its double score 1.0 and vectors at right angles score 0.

**Before you move on**, check that you can:

- work out the cosine similarity of two short vectors by hand;
- say what a cosine similarity of 1, 0 and -1 means;
- say why `[3, 4]` and `[6, 8]` score exactly 1;
- say why one number of a learned embedding cannot be read on its own.

## 2. Vector similarity search

**Vector similarity search** turns every stored text into a vector with one model, turns the question into a vector with the same model, and returns the stored texts whose vectors are most similar to the question's. This section does it with NumPy alone, on 30 short articles.

### 2.1 A model turns text into numbers

A small model turns a text into 384 numbers; Thursday opens it. It is `all-MiniLM-L6-v2` from the sentence-transformers library, which Colab ships. It maps a sentence or a short paragraph to 384 numbers, it is built for search and for grouping similar texts, and it cuts off input longer than 256 word pieces.

- `encoder` is the model, loaded once. The first run downloads its files from Hugging Face, a public site that hosts models, so it takes a moment. On Colab the first load may also print a warning about unauthenticated requests to the HF Hub; it is harmless, and the model loads as normal.
- `embed(texts)` takes a list of texts and returns one row of 384 numbers per text. `normalize_embeddings=True` scales each row to length 1, and `float32` stores each number in 4 bytes.

Every vector today, in SQLite, in Chroma and in Pinecone, comes from `embed`.

**Predict first:** one text goes in, as a list of one. What shape comes out?

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # downloads once, then runs on the CPU

def embed(texts):
    return encoder.encode(texts, normalize_embeddings=True).astype("float32")

hello = embed(["hello"])
print(hello.shape, hello.dtype)
print("first five numbers:", np.round(hello[0, :5], 3))
print(f"length {np.linalg.norm(hello[0]):.3f}")

- `(1, 384)`: one row per text, 384 numbers per row, stored as `float32`.
- The first five numbers of "hello" are about -0.063, 0.055, 0.052, 0.086 and -0.083. None of them has a name, unlike action and romance in 1.1.
- The length prints 1.000, so a dot product of two rows from `embed` is their cosine similarity.
- **Common mistake:** embedding a question with a different model, or with `encoder.encode` and no `normalize_embeddings=True`. A vector only has a meaning next to vectors from the same model, and without length 1 the dot product is no longer the cosine similarity.

**In an interview:** to vectorise text, run it through an embedding model such as all-MiniLM-L6-v2, which returns 384 numbers per text, placed so that texts about the same thing land close together. The same model must embed the stored texts and the questions.

### 2.2 The help-centre articles

`KB` holds 30 short help-centre articles for an online shop. They are **invented** for this notebook, so every record can be read whole. Each is a dictionary with six keys:

- `id` (`"kb-001"` to `"kb-030"`) and `title`;
- `text`, one or two sentences: the part that is embedded;
- `category`: `"account"`, `"billing"`, `"shipping"` or `"technical"`;
- `product`: `"web"` or `"mobile"`;
- `updated`: the year the article last changed, 2023 to 2026.

Every later section stores this list. kb-003 is the record the update cells change, and kb-030 the one the delete cells remove. The second cell prints the list as a table, then counts the articles per category and per product.

**Predict first:** how many rows will the table print, and how many categories will the count list?

In [ ]:
# Invented help-centre articles, one per line: id, title, text, category, product, year last updated
KB = [
    {"id": "kb-001", "title": "Resetting a forgotten password", "text": "Choose Forgot password on the sign-in screen, type the email address you registered with, and open the link we send to pick a new password. The link expires after 30 minutes.", "category": "account", "product": "web", "updated": 2025},
    {"id": "kb-002", "title": "Turning on two-step verification", "text": "Two-step verification asks for a six-digit code from an authenticator app each time you sign in on a new device. Turn it on under Settings, Security, then scan the QR code shown.", "category": "account", "product": "web", "updated": 2024},
    {"id": "kb-003", "title": "Changing the email on your account", "text": "Open Settings, then Profile, and enter the new email address. We send a confirmation link to the new address, and the change takes effect only after you open that link.", "category": "account", "product": "web", "updated": 2023},
    {"id": "kb-004", "title": "Deleting your account", "text": "Deleting your account removes your profile, saved addresses and order history after a 14-day grace period. Request it under Settings, Privacy, Delete account; signing in again during the grace period cancels the request.", "category": "account", "product": "web", "updated": 2024},
    {"id": "kb-005", "title": "Signing out on every device", "text": "If you used a shared or lost phone, choose Sign out everywhere under Settings, Security. Every session ends at once, and each device asks for your password the next time it opens.", "category": "account", "product": "mobile", "updated": 2025},
    {"id": "kb-006", "title": "Signing in with fingerprint or face", "text": "On phones that support it, turn on biometric sign-in under Settings, Security in the mobile app. Your fingerprint or face data stays on the phone and is never sent to our servers.", "category": "account", "product": "mobile", "updated": 2026},
    {"id": "kb-007", "title": "Updating your display name", "text": "Your display name appears on reviews and messages. Change it under Settings, Profile, Display name; the new name shows everywhere within a few minutes and you may change it once every 30 days.", "category": "account", "product": "mobile", "updated": 2023},
    {"id": "kb-008", "title": "Ending a plan", "text": "To end a plan, open Billing, choose Manage plan and select Cancel renewal. Access continues until the current period ends, and no further amount is billed after that date.", "category": "billing", "product": "web", "updated": 2025},
    {"id": "kb-009", "title": "Two entries for one purchase", "text": "If your card statement lists the same purchase two times, the second line is usually a temporary authorisation hold. Your bank releases it within five business days; contact us if it remains after that.", "category": "billing", "product": "web", "updated": 2024},
    {"id": "kb-010", "title": "Downloading an invoice", "text": "Every order has a PDF invoice with tax shown separately. Open Orders, select the order, and choose Download invoice; invoices for business purchases also list your company name and tax number.", "category": "billing", "product": "web", "updated": 2026},
    {"id": "kb-011", "title": "Updating a payment card", "text": "Add or replace a card under Billing, Payment methods. The new card becomes the default for future orders and renewals, and an expired card is removed automatically after its last valid month.", "category": "billing", "product": "web", "updated": 2023},
    {"id": "kb-012", "title": "Refund timelines", "text": "Once a return is approved, the refund goes back to the original payment method. Cards usually show it within 5 to 10 business days; store credit appears in your wallet immediately.", "category": "billing", "product": "mobile", "updated": 2025},
    {"id": "kb-013", "title": "Applying a promo code", "text": "Enter a promo code in the Discount box at checkout before you pay. Only one code applies per order, codes cannot be added after an order is placed, and some exclude sale items.", "category": "billing", "product": "mobile", "updated": 2024},
    {"id": "kb-014", "title": "Paying in instalments", "text": "Orders over 100 dollars can be split into four equal instalments, taken every two weeks from the same card. Choose Pay in four at checkout; a missed instalment is retried after three days.", "category": "billing", "product": "mobile", "updated": 2026},
    {"id": "kb-015", "title": "Tracking an order that has not arrived", "text": "If a package has not arrived ten business days after dispatch, open Orders, select it and choose Report a problem. We trace it with the courier, then resend the items or refund you.", "category": "shipping", "product": "web", "updated": 2025},
    {"id": "kb-016", "title": "Delivery times and costs", "text": "Standard delivery takes three to seven business days and is free over 50 dollars. Express delivery arrives in one to two business days for a flat fee shown at checkout before you pay.", "category": "shipping", "product": "web", "updated": 2024},
    {"id": "kb-017", "title": "Changing a delivery address", "text": "You can change the delivery address while an order shows Processing. Open Orders, choose Edit address and save; once the status changes to Dispatched, contact the courier using the tracking number instead.", "category": "shipping", "product": "web", "updated": 2023},
    {"id": "kb-018", "title": "Shipping outside Canada", "text": "We ship to the United States and the United Kingdom. Import duties and taxes are collected by the courier on delivery and are not included in the price shown at checkout.", "category": "shipping", "product": "web", "updated": 2026},
    {"id": "kb-019", "title": "Returning an item", "text": "Most items can be returned within 30 days of delivery. Start a return under Orders, print the prepaid label, and drop the box at any post office; we email you when it is received.", "category": "shipping", "product": "mobile", "updated": 2024},
    {"id": "kb-020", "title": "Damaged items on arrival", "text": "If something arrives broken, take a photo of the item and the box within 48 hours and attach both to a problem report under Orders. We replace it at no cost without needing it back.", "category": "shipping", "product": "mobile", "updated": 2025},
    {"id": "kb-021", "title": "Picking up from a locker", "text": "Choose a pickup locker at checkout to collect your order yourself. You receive a code by text message when it is ready, and the locker holds it for five days before it is returned.", "category": "shipping", "product": "mobile", "updated": 2026},
    {"id": "kb-022", "title": "Fixing crashes on phones", "text": "If the mobile version quits unexpectedly, install the latest update from your phone's store, restart the phone, and clear cached data under Settings. Send a report from the Help menu if it continues.", "category": "technical", "product": "mobile", "updated": 2025},
    {"id": "kb-023", "title": "Turning on notifications", "text": "Allow notifications for our mobile app in your phone's system settings, then choose which alerts you want under Settings, Notifications: order updates, price drops and messages can each be switched separately.", "category": "technical", "product": "mobile", "updated": 2024},
    {"id": "kb-024", "title": "Supported browsers", "text": "The website supports the two latest versions of Chrome, Firefox, Safari and Edge. Older browsers may load pages slowly or show broken layouts, so update the browser before reporting a display problem.", "category": "technical", "product": "web", "updated": 2023},
    {"id": "kb-025", "title": "Pages loading slowly", "text": "Slow pages are usually caused by a weak network connection or a full browser cache. Try another network, clear the cache and cookies, and turn off extensions that block scripts before contacting support.", "category": "technical", "product": "web", "updated": 2024},
    {"id": "kb-026", "title": "Working offline in the mobile app", "text": "The mobile app keeps your saved lists and recent orders on the phone, so you can read them without a connection. Changes made offline sync automatically the next time the phone is online.", "category": "technical", "product": "mobile", "updated": 2026},
    {"id": "kb-027", "title": "Exporting your data", "text": "Request a copy of everything we store about you under Settings, Privacy, Export data. We prepare a ZIP file of JSON and CSV files and email a download link within 48 hours.", "category": "technical", "product": "web", "updated": 2025},
    {"id": "kb-028", "title": "Connecting a smart speaker", "text": "Link a smart speaker under Settings, Connected devices, then say the wake word followed by order status to hear where your latest order is. Unlink the speaker from the same page at any time.", "category": "technical", "product": "mobile", "updated": 2026},
    {"id": "kb-029", "title": "Error codes at checkout", "text": "Error E102 means the card issuer declined the payment and E205 means the session timed out. Refresh the page and try again, or choose another payment method if the same code repeats.", "category": "technical", "product": "web", "updated": 2023},
    {"id": "kb-030", "title": "Using the site with a screen reader", "text": "Every page works with common screen readers such as NVDA, JAWS and VoiceOver. Use the Skip to content link at the top of each page, and report any control that is not read aloud correctly.", "category": "technical", "product": "web", "updated": 2024},
]

In [ ]:
kb_df = pd.DataFrame(KB)
print(kb_df[["id", "title", "category", "product", "updated"]].to_string(index=False))
print(kb_df["category"].value_counts().to_dict())
print(kb_df["product"].value_counts().to_dict())

- 30 rows in four categories: account 7, billing 7, shipping 7 and technical 9; web 17 and mobile 13.
- Read two of them: kb-001 "Resetting a forgotten password" and kb-008 "Ending a plan". Both come back in 2.3, found by questions that share none of their words.
- `title`, `category`, `product` and `updated` are what a search shows or filters on. Only `text` becomes a vector.
- **Common mistake:** rewording an article here. The searches in 2.3 were checked on this exact text, and a reworded article can lose its match.

**In an interview:** each record for a vector store has an id, the text that gets embedded, and metadata fields beside it (here category, product and year) that a search can show or filter on.

### 2.3 Search by comparing with every row

To search, embed the 30 texts once, embed the question, and score the question against every article.

- `kb_vectors = embed([a["text"] for a in KB])` has one row per article: shape `(30, 384)`.
- `query_vector = embed([query])[0]` embeds a list of one text, then keeps its only row: shape `(384,)`. Every later section uses this spelling.
- `kb_vectors @ query_vector` (`@` is matrix multiplication) dots each of the 30 rows with the question and gives 30 scores. For vectors of length 1 the dot product is the cosine similarity.
- `np.argsort` returns positions from the smallest value to the largest, so `np.argsort(-scores)[:3]` puts the largest first and keeps three. These three are the question's **nearest neighbours**: the stored vectors most similar to it.

The question is "I can't get into my account".

**Predict first:** will the top article share a word with the question?

In [ ]:
kb_vectors = embed([a["text"] for a in KB])           # one row of 384 numbers per article
query = "I can't get into my account"                 # Try changing it: a question of your own
query_vector = embed([query])[0]                      # the only row: shape (384,)
scores = kb_vectors @ query_vector                    # length-1 vectors: dot product = cosine similarity
top = np.argsort(-scores)[:3]                         # the minus sign puts the largest score first
print(f"kb_vectors {kb_vectors.shape}, query_vector {query_vector.shape}")
for i in top:
    print(f"{KB[i]['id']}  {scores[i]:.2f}  {KB[i]['title']}")

- kb-001 "Resetting a forgotten password" comes first at about 0.46, then kb-004 "Deleting your account" at about 0.33 and kb-005 "Signing out on every device" at about 0.32.
- kb-001 shares no word with the question, in its title or its text. The model placed the two texts close because they are about the same problem.
- Read the order and the gap, not the size. Related short texts score between about 0.3 and 0.65 with this model, nowhere near 1.0, and 0.46 stands well clear of 0.33. A rule such as "trust only scores above 0.8" would throw away every good match here.
- Search has no threshold built in. A question about baking bread still gets three articles back, all scoring about 0.1 or less, and only the scores show they are poor.
- **Common mistake:** `np.argsort(scores)[:3]` without the minus sign. It sorts from the smallest up, so it returns the three worst articles, with no error.
- Try changing it: set `query` to a question of your own, rerun, and read whether the top article shares any word with it. Four more questions that share no word with the article they find: "my parcel never showed up" (kb-015 first, about 0.56), "why was I charged twice" (kb-009, about 0.32), "the app keeps closing by itself" (kb-022, about 0.50) and "how do I stop paying every month" (kb-008, about 0.55). Then put `"I can't get into my account"` back and rerun the cell: every later section reads `query_vector`.

**In an interview:** similarity search embeds the stored texts and the question with the same model, scores the question against every stored vector (a dot product, which is the cosine similarity for vectors of length 1) and returns the top k, nearest first. It matches by meaning, so no shared word is needed.

### 2.4 A similarity heat map

*Going deeper. The one idea to keep: articles about the same thing score higher with each other than with the rest.*

Six articles, two each from account, billing and shipping, scored against each other at once: their six rows of `kb_vectors` times the same rows turned on their side (`.T`) give a 6-by-6 table of cosine similarities, drawn by `plot_similarity` from the setup cell.

In [ ]:
six = ["kb-001", "kb-005", "kb-008", "kb-009", "kb-015", "kb-016"]
kb_ids = [a["id"] for a in KB]
rows = [kb_ids.index(kb_id) for kb_id in six]
sims = kb_vectors[rows] @ kb_vectors[rows].T          # every pair at once
plot_similarity(sims, [f"{KB[i]['id']} {KB[i]['title'][:22]}" for i in rows],
                title="Cosine similarity of six articles")

- The diagonal is 1.00: each article against itself.
- The brightest square off the diagonal pairs the two account articles, kb-001 and kb-005 (about 0.60). The two shipping articles, kb-015 and kb-016, come next among the pairs that share a category (about 0.39).
- A category is a label people chose, and the vectors do not always agree with it: billing's kb-009 "Two entries for one purchase" scores about 0.36 with shipping's kb-015, a little higher than with its billing partner kb-008 (about 0.30). Both talk about an order that went wrong.
- The map is the same on both sides of the diagonal: A against B is the same dot product as B against A.
- **Common mistake:** reading a pale square as "opposite". It means a low similarity, not much in common. Opposite would be near -1; the lowest pair here, kb-005 against kb-016, is about -0.05, which is no different from 0 in practice.
- Two different texts with a very bright square between them are near-duplicates, which is one of the use cases in 4.2.

**Before you move on**, check that you can:

- say what shape `embed` returns for one text and for 30;
- write the three lines that score a question against stored vectors and keep the top 3;
- say why `np.argsort(-scores)` needs the minus sign;
- say why a top score of 0.46 can still be a good match, and why a search always returns something.

## 3. Storing and retrieving data in SQLite

Before a vector database, an ordinary one. **SQLite** is a whole database in one file with no separate server, and Python ships the `sqlite3` module, so week 3's SQL now runs from Python.

### 3.1 A table with a vector column

The table is `articles (id TEXT PRIMARY KEY, title TEXT, text TEXT, category TEXT, product TEXT, updated INTEGER, vector BLOB)` in the file `kb.db`.

- A relational table has no vector type, but a **BLOB** column holds raw bytes, which arrive in Python as a `bytes` value.
- `vector.tobytes()` copies the array's raw bytes, so the whole vector fits in one column.
- Each `?` is a **placeholder**: the values travel apart from the SQL text, so a value can never change the statement.
- `executemany` runs the one `INSERT` once per row, and `commit()` saves the transaction the insert opened. Without it the rows are lost when the connection closes.
- `DROP TABLE IF EXISTS` comes first, so the cell can run twice.

**Predict first:** each vector is 384 numbers of 4 bytes each. How many bytes will one vector take in the `BLOB` column?

In [ ]:
con = sqlite3.connect("kb.db")                        # one file in Colab's working folder
con.execute("DROP TABLE IF EXISTS articles")          # so the cell reruns cleanly
con.execute("CREATE TABLE articles (id TEXT PRIMARY KEY, title TEXT, text TEXT, category TEXT, "
            "product TEXT, updated INTEGER, vector BLOB)")
rows = [(a["id"], a["title"], a["text"], a["category"], a["product"], a["updated"], v.tobytes())
        for a, v in zip(KB, kb_vectors)]
con.executemany("INSERT INTO articles VALUES (?, ?, ?, ?, ?, ?, ?)", rows)   # values travel apart from the SQL
con.commit()
print(con.execute("SELECT COUNT(*) FROM articles").fetchone())
print(f"{len(rows[0][-1])} bytes per vector")

- `(30,)`: the count comes back as a one-item tuple, one row per article.
- 1,536 bytes per vector: 384 numbers times 4 bytes.
- `kb.db` lives in Colab's working folder, `/content`. It survives a runtime restart but not a new runtime.
- **Common mistake:** building the `INSERT` with an f-string that pastes the values into the SQL text. It runs on these articles, and it is how **SQL injection** happens: a crafted value rewrites the statement. With placeholders the value never becomes part of the statement.

**In an interview:** a relational table can hold a vector as bytes in a `BLOB` column, written with `tobytes()` through a parameterised `INSERT`, with the text and metadata in ordinary columns beside it.

### 3.2 Reading vectors back

`np.frombuffer(blob, dtype=np.float32)` turns the bytes back into an array. Its `dtype` defaults to `float64`, eight bytes per number, while the vectors were written as `float32`, four bytes per number. The first cell reads kb-001's vector both ways.

**Predict first:** the 1,536 bytes of one vector are read back with the default `dtype`. How many numbers come out?

In [ ]:
blob = con.execute("SELECT vector FROM articles WHERE id = ?", ("kb-001",)).fetchone()[0]
wrong = np.frombuffer(blob)                           # default dtype float64: 8 bytes per number
right = np.frombuffer(blob, dtype=np.float32)         # the dtype the vector was written in
print(len(blob), wrong.shape, right.shape)
print("same as kb_vectors[0]:", np.allclose(right, kb_vectors[0]))

- `1536 (192,) (384,)`: with the default `dtype` the bytes become 192 meaningless numbers, and nothing raises an error. With `dtype=np.float32` the original 384 come back unchanged.
- **Common mistake:** `np.frombuffer(blob)` with no `dtype`. Bytes are just bytes and NumPy reads them as whatever it is told, so the mistake only shows later, when a 192-number vector meets the 384-number question and the product fails.

Now a search straight from the table: read every row, stack the vectors into one 30-by-384 array with `np.vstack`, score them against the question, and take the position of the highest score with `np.argmax`.

**Predict first:** will the best article be the same one 2.3 found?

In [ ]:
ids, vectors = [], []
for id_, blob in con.execute("SELECT id, vector FROM articles"):   # reads every row
    ids.append(id_)
    vectors.append(np.frombuffer(blob, dtype=np.float32))
sql_scores = np.vstack(vectors) @ query_vector
best = np.argmax(sql_scores)
print(f"{len(ids)} rows read and compared; best {ids[best]} at {sql_scores[best]:.2f}")

- 30 rows read, and the best is kb-001 at about 0.46, as in 2.3: the vectors went into the file and came back unchanged.
- To find one article the loop read and compared every row. Hold on to that for 3.3.

A `WHERE` clause narrows the rows before any comparison. The next cell asks a billing question, "why was I charged twice", of the billing articles only.

In [ ]:
billing_vector = embed(["why was I charged twice"])[0]
billing = con.execute("SELECT id, title, vector FROM articles WHERE category = 'billing'").fetchall()
billing_scores = [float(np.frombuffer(blob, dtype=np.float32) @ billing_vector) for _, _, blob in billing]
print(f"{len(billing)} billing rows compared")
for (id_, title, _), score in sorted(zip(billing, billing_scores), key=lambda pair: -pair[1])[:3]:
    print(f"{id_}  {score:.2f}  {title}")

- 7 billing rows compared, not 30. kb-009 "Two entries for one purchase" comes first at about 0.32, well clear of kb-014 at about 0.21, again with no word shared with the question.
- Narrowing by a field before comparing vectors is what a vector database's **metadata filter** does at scale. Its operators are Wednesday's.

**In an interview:** to search vectors kept in SQL, read each vector back with `np.frombuffer` and the `dtype` it was written in, score every row against the question and keep the best. A `WHERE` clause can narrow the rows first, but the comparison still visits every remaining row.

### 3.3 What happens at 50,000 rows

The first cell fills a second table, `scale_test`, with `N_ROWS = 50_000` random vectors scaled to length 1: `axis=1` takes one length per row, and `keepdims=True` keeps the lengths as a column so each row is divided by its own length. Random vectors are enough, because the time does not depend on what the vectors mean, only on how many there are.

The second cell times a **brute-force search**: read every blob, score it against `query_vector`, keep the best. SQLite has no index for vectors, so every search here is brute force. The cell then drops the table and runs `VACUUM`, which gives the file's space back to Colab's disk.

**Predict first:** if the table had twice as many rows, what would happen to the search time?

In [ ]:
N_ROWS = 50_000                                       # Try changing it: 100_000
rng = np.random.default_rng(RANDOM_STATE)
random_vectors = rng.normal(size=(N_ROWS, 384)).astype("float32")
random_vectors /= np.linalg.norm(random_vectors, axis=1, keepdims=True)   # every row to length 1
con.execute("DROP TABLE IF EXISTS scale_test")
con.execute("CREATE TABLE scale_test (id INTEGER PRIMARY KEY, vector BLOB)")
start = time.perf_counter()
con.executemany("INSERT INTO scale_test VALUES (?, ?)",
                ((i, v.tobytes()) for i, v in enumerate(random_vectors)))
con.commit()
print(f"inserted {N_ROWS:,} rows in {time.perf_counter() - start:.1f} s; "
      f"kb.db is now about {os.path.getsize('kb.db') / 1e6:.0f} MB")

In [ ]:
start = time.perf_counter()
best_score, best_id = -1.0, None
for row_id, blob in con.execute("SELECT id, vector FROM scale_test"):   # every row, one by one
    score = float(np.frombuffer(blob, dtype=np.float32) @ query_vector)
    if score > best_score:
        best_score, best_id = score, row_id
print(f"searched {N_ROWS:,} rows in {time.perf_counter() - start:.2f} s; best row {best_id} at {best_score:.2f}")

con.execute("DROP TABLE scale_test")                  # give the disk space back
con.commit()
con.execute("VACUUM")
print(f"kb.db is back to about {os.path.getsize('kb.db') / 1e6:.2f} MB")

- On our Colab run the insert took about 0.8 seconds and the search about 0.36 seconds, with the file at about 103 MB. Your times depend on the machine and change a little from run to run; the shape of the result does not.
- The best score is low, about 0.2: random vectors mean nothing, so none is close to the question. Only the time matters here.
- A fraction of a second is fine once. It is not fine when every search pays it, on millions of rows, for many users at once. Avoiding that full comparison is why vector databases exist.
- Try changing it: `N_ROWS = 100_000` in the first cell, then run both cells. The search time roughly doubles (in a laptop check of the same code, about 0.35 seconds at 50,000 rows and about 0.66 at 100,000; this size was not timed on Colab), the insert takes longer and the file reaches about 205 MB before the drop. Put it back to `50_000` and run both cells again.
- **Common mistake:** leaving `100_000` in place and running the cells again and again, or stopping the second cell partway. Each run writes about 200 MB, and a stopped cell never reaches the `DROP TABLE`, so the table stays on disk until the second cell runs to the end.

**In an interview:** SQLite can store vectors, but it has no vector index, so every search reads and compares every row. The time grows in step with the row count: twice the rows, about twice the time. A vector database's index exists to avoid that full scan.

**Before you move on**, check that you can:

- write the `CREATE TABLE` line for a table with a vector column, and say why the column is a `BLOB`;
- read a stored vector back with the right `dtype`, and say what the default gives;
- say why the `INSERT` uses `?` placeholders;
- say what happens to a brute-force search's time when the rows double, and why.

## 4. Vector databases

### 4.1 What a vector database adds

A **vector database** indexes and stores vector embeddings for fast retrieval and similarity search. A traditional database looks for rows whose values exactly match the query; a vector database applies a similarity measure to find the vectors most like the query's. Against what section 3 lacked, it adds five things:

- **An index that finds near vectors without reading every row.** An **approximate nearest-neighbour (ANN) index** reaches the vectors close to a query without comparing it with every stored one. The best-known kind is **HNSW**, a graph that links each vector to its near neighbours, and it is the one Chroma builds. Product quantization and locality-sensitive hashing are two others. It is the same idea as week 3's SQL index: a B-tree finds rows with an exact value without reading the whole table, and an HNSW graph finds vectors near a query without visiting every row.
- **Metadata kept and filtered beside each vector**, the job 3.2's `WHERE` did by hand.
- **Writes while serving**: new records go in while queries keep running.
- **Scale** past one machine.
- **Client libraries** that connect, write and query, so no SQL and no byte conversion is written by hand.

"Approximate" has a price: the index can miss a few of the true nearest neighbours. It gives up a little **recall** (the share of the true neighbours it finds) for a large gain in speed, and teams accept that because a near-perfect answer now beats a perfect one after reading millions of rows. One limit: an HNSW index must sit in memory to be queried or updated, so one machine's RAM caps a single Chroma collection.

**Predict first:** SQLite read every row in 3.3. What does a vector database do instead, and what does it give up for it?

<details>
<summary>Check your answer</summary>

It keeps an index, such as an HNSW graph, that reaches the near vectors without visiting every row. It gives up a little recall: the index can miss a few of the true nearest neighbours. Everything it returns is a real, close vector; it may just leave out one that a full comparison would have found.

</details>

The tools you will meet:

| Tool | What it is | Where it runs |
|---|---|---|
| Chroma | open-source vector database | inside your Python program (as today), or as a server |
| Pinecone | managed vector database | on Pinecone's servers, as serverless indexes; you reach it with an API key |
| pgvector | vector similarity search inside PostgreSQL, with HNSW and IVFFlat indexes | in your PostgreSQL database |
| Qdrant | open-source vector database | your own server, or its managed cloud |
| Weaviate | open-source vector database | its cloud, Docker or Kubernetes |
| Milvus | open-source vector database | from Milvus Lite through Standalone to Distributed |
| FAISS | a library for similarity search of dense vectors, not a server | inside your program, with no metadata store of its own |

- Today runs Chroma and Pinecone: one fits inside a notebook, the other is fully managed. The ideas carry over to the rest.
- **Common mistake:** hearing "approximate" as "wrong". The index may leave out a few true neighbours, but the results it returns are real, close vectors.

**In an interview:** a vector database stores embeddings with their metadata and answers similarity queries through an approximate nearest-neighbour index such as HNSW, so it does not compare the query with every row. It trades a little recall for speed, and adds filtering, writes while serving, scale and client libraries.

### 4.2 Use cases

Every use case below has one shape: turn the items into vectors with one model, turn the question or the item in hand into a vector with the same model, and return the nearest stored ones. Of any app that finds "things like this one", ask two questions: what are the vectors made from, and what does the metadata hold?

**Work it by hand first.** Name one app you use that finds "things like this one": a playlist, a shop's "similar items" row, a help-centre search box. Write one line before you read on:

```
App: ...   Vectors made from: ...   Metadata holds: ...
```

<details>
<summary>Check your answer</summary>

Find the row of the table below that your app fits; an app that fits no row goes under the nearest one. A good line names something concrete for each part: "Music app: vectors made from the songs' audio or descriptions; metadata holds the genre, the year and whether it is explicit."

Two usual wrong answers, and what causes each:

- "Vectors made from what other people bought." That is a table of who bought what, week 7's recommender. A similarity recommender compares the items' own vectors, made from their descriptions or photos.
- "A better model" as the fix when a help-centre search shows a web-site article to someone on the mobile app. The model found an alike article, which is its job. Whether the article applies to this user is a question for the metadata: filter on `product`.

</details>

| Use case | Vectors made from | Metadata holds |
|---|---|---|
| Help-centre or document search | articles | category, product |
| Recommendations | products or films a user liked | genre, price |
| Duplicate detection | support tickets | status, date |
| Image search | photos | size, owner |
| Unusual-item flags | transactions | account, amount |
| Retrieval for a language model's answer | document passages | source, date |

- **Help-centre search** is this morning's own example, with category and product as metadata.
- **Recommendations:** week 7's recommender filled the blanks of a ratings table. A similarity recommender returns items whose vectors sit near the ones a user already liked.
- **Duplicate detection:** a new ticket whose vector sits very close to an open one is likely a duplicate, and the status keeps the search to open tickets.
- **Image search** builds its vectors from photos instead of text, and the same nearest-neighbour search runs on them. An **unusual-item flag** looks for the opposite: an item whose vector sits far from every usual one.
- **Retrieval for a language model's answer**, called **retrieval-augmented generation (RAG)**: the application embeds the question, retrieves similar stored text from a vector database and gives both to a language model, which writes the answer. It is named here only. This week builds the retrieval half, and week 9 adds the model.

**Predict first:** which of the six rows needs a language model to write anything?

<details>
<summary>Check your answer</summary>

Only the last one, retrieval for a language model's answer. The other five are search, ranking and flags: an embedding model makes the vectors, but nothing in them generates text.

</details>

- **Common mistake:** treating the nearest match as the right answer. Similarity says two things are alike, not that the match is correct, current or allowed for this user, so real searches also filter on metadata such as dates, product or permissions.

**In an interview:** vector search powers semantic search, recommendations, duplicate detection, image search, unusual-item flags and retrieval for a language model's answer. For each, say what the vectors are made from and what the metadata filters on, because similarity alone finds what is alike, not what is allowed.

**Before you move on**, check that you can:

- name three things a vector database adds over an SQLite table;
- say what "approximate" costs in an approximate nearest-neighbour index;
- name three use cases and, for each, what the vectors and the metadata are made from;
- say which use case needs a language model, and which week builds it.

## 5. Chroma DB setup

**Chroma** is an open-source vector database that runs inside your Python program, here inside the notebook, and can also run as a server that several programs share. It needs no account and no key.

### 5.1 Installing Chroma

- `%pip install` runs pip, Python's package installer, for the notebook's own Python, and `-q` keeps its output short.
- `==1.5.9` pins the version, so the calls in this week's notebooks keep working when a newer release comes out. Colab does not ship Chroma.
- The install adds about fifteen packages Colab lacks, onnxruntime among them, and upgrades three that Colab already has (opentelemetry-api, opentelemetry-sdk and oauthlib on our Colab run). Colab did not ask for a runtime restart.
- `if IN_COLAB:` (set in the setup cell) runs the line on Colab only. A Python outside Colab that already has chromadb 1.5.9 skips it.

**Predict first:** will Colab ask to restart the runtime after this install?

In [ ]:
if IN_COLAB:
    %pip install -q chromadb==1.5.9
import chromadb

print(f"chromadb {chromadb.__version__}")

- `chromadb 1.5.9`, and no restart prompt: the install added packages without changing any that were already loaded.
- Colab may also print red text that starts `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed` and names packages such as google-adk and tweepy. The install still succeeds, and this notebook does not use those packages.
- If Colab ever does ask to restart, restart once, click the first cell of 5.2 and choose Runtime > Run before.
- A new runtime starts again from Colab's own packages, so every notebook this week installs the pinned version again.
- **Common mistake:** typing `pip install chromadb` with no pin in a cell of your own. You get the newest release, whose calls may differ from the ones this notebook makes.

**In an interview:** install Chroma with a pinned version (`%pip install -q chromadb==1.5.9` in a notebook), so every run gets the same behaviour. It then runs inside the Python program, with no server to set up.

### 5.2 Clients

A **client** is the object your code talks to, and the constructor you call decides where the data lives:

- `chromadb.EphemeralClient()` keeps everything in memory. It is gone when the runtime restarts: fine for a throwaway test.
- `chromadb.PersistentClient(path="chroma_kb")` keeps its data in the folder `chroma_kb` on disk and loads it again when a client opens that folder. On Colab the folder survives a runtime restart but not a new runtime.
- `chromadb.HttpClient(host=..., port=...)` reaches a Chroma server and `CloudClient` reaches Chroma's hosted service. They are named here only; the notes page shows the server.
- `heartbeat()` returns the client's clock in nanoseconds, a 19-digit number. Its job is to show the client is answering.

The first two lines make the notebook rerun cleanly from the top. Chroma keeps one open store per folder for as long as the Python process runs, so the first line forgets any store an earlier run opened, and the second removes the folder. Without the first line, a second Runtime > Run all fails in 6.1 with "Collection [kb_articles] already exists", even after the folder is gone.

**Predict first:** after a runtime restart, which of the two clients still has its data?

In [ ]:
chromadb.api.client.SharedSystemClient.clear_system_cache()   # forget stores an earlier run opened
shutil.rmtree("chroma_kb", ignore_errors=True)   # start from an empty store, so the notebook reruns from the top

scratch = chromadb.EphemeralClient()                    # in memory: gone when the runtime restarts
client = chromadb.PersistentClient(path="chroma_kb")    # on disk, in the folder chroma_kb
print(f"ephemeral heartbeat  {scratch.heartbeat()}")
print(f"persistent heartbeat {client.heartbeat()}")
print("chroma_kb folder exists:", os.path.isdir("chroma_kb"))

- Two 19-digit numbers: both clients are answering.
- The folder `chroma_kb` now exists. It is why the `PersistentClient` still has its data after a restart, while the `EphemeralClient` keeps nothing outside memory.
- Use `EphemeralClient` for a throwaway test, `PersistentClient` when one program keeps its data in a folder, and `HttpClient` when several programs share one Chroma server.
- **Common mistake:** deleting the two lines at the top as clutter. The first Run all still works; the second fails in 6.1 with "Collection [kb_articles] already exists".

**In an interview:** Chroma's client decides where the data lives: `EphemeralClient` in memory for tests, `PersistentClient(path=...)` in a folder on disk, and `HttpClient` for a Chroma server that several programs share.

### 5.3 Chroma keeps its records in SQLite

*Going deeper. The one idea to keep: a vector database is ordinary storage plus an index.*

Chroma's single-node store is SQLite again. `chroma_kb/` holds a file `chroma.sqlite3`, which keeps the documents and metadata, and, once records are added, one folder per collection for its HNSW index. The cell opens the file read-only with `sqlite3` and lists its tables.

In [ ]:
chroma_file = sqlite3.connect("file:chroma_kb/chroma.sqlite3?mode=ro", uri=True)   # read-only
tables = [name for (name,) in chroma_file.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")]
chroma_file.close()
print(f"{len(tables)} tables, among them:", [t for t in tables if t in
      ("collections", "embeddings", "embedding_metadata", "segments", "embeddings_queue")])
print("in the folder:", sorted(os.listdir("chroma_kb")))

- 21 tables, among them `collections`, `embeddings`, `embedding_metadata`, `segments` and `embeddings_queue`.
- The folder holds only `chroma.sqlite3` so far. The collection folder, named with a long random id (a UUID), appears once 6.1 adds records; it holds the HNSW index, which is loaded into memory while the collection is used.
- **Common mistake:** writing code that reads `chroma.sqlite3` directly. The layout is Chroma's own and may change between releases; read and write through the client.

**Before you move on**, check that you can:

- write the pinned install line for Chroma, and say why it is pinned;
- choose between `EphemeralClient` and `PersistentClient` for a throwaway test and for data that must survive a restart;
- say what the two lines at the top of 5.2 prevent;
- name the client that reaches a Chroma server.

## 6. A local vector database with Chroma

A **collection** is a named set of records, roughly what a table is in SQL. Each record has an id, a document (the text), an embedding (its vector) and metadata.

### 6.1 A collection

`client.create_collection("kb_articles", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})`:

- `embedding_function=None` switches off Chroma's own embedding. A collection made without it gets Chroma's default embedding function: its own copy of all-MiniLM-L6-v2, run through ONNX and downloaded on first use, which would quietly embed any text sent without a vector. Switched off, the collection holds exactly the vectors `embed` made, so SQLite, Chroma and Pinecone hold identical vectors and their results compare row for row.
- `configuration={"hnsw": {"space": "cosine"}}` compares vectors by cosine. The default space is squared L2, a straight-line distance; comparing the two is Wednesday's.
- `add(ids=..., documents=..., embeddings=..., metadatas=...)` takes four lists that line up by position: the first id goes with the first document, the first vector and the first metadata. Metadata values may be strings, integers, floats, booleans or non-empty lists of one type.
- `count()` is the check after every write. An `add` with an id that already exists is skipped with no error and the old record stays, so only the count shows it.

**Predict first:** we add 30 records. What will `count()` print?

In [ ]:
collection = client.create_collection(
    "kb_articles",
    embedding_function=None,                          # every vector comes from embed, none from Chroma
    configuration={"hnsw": {"space": "cosine"}},      # compare by cosine; the default is squared L2
)
collection.add(
    ids=[a["id"] for a in KB],
    documents=[a["text"] for a in KB],
    embeddings=kb_vectors,
    metadatas=[{"title": a["title"], "category": a["category"], "product": a["product"],
                "updated": a["updated"]} for a in KB],
)
print(f"{collection.name}: {collection.count()} records")

- `kb_articles: 30 records`.
- A vector of the wrong length is refused: 10 numbers sent to this 384-number collection raise `InvalidArgumentError: Collection expecting embedding with dimension of 384, got 10`.
- **Common mistake:** leaving out `embedding_function=None`. The cell still works, but Chroma attaches its own model, downloads it, and from then on embeds any text sent without a vector with a second model, so the stores no longer hold the same vectors.

**In an interview:** a Chroma collection is a named set of records, each with an id, a document, an embedding and metadata, compared in the space set at creation (cosine here). With `embedding_function=None` it stores only the vectors you send, so every vector comes from one model you chose.

### 6.2 Query

`collection.query(query_embeddings=[query_vector], n_results=3)` ranks the records by nearest neighbour. Without `n_results` it returns 10.

- The result is a dictionary of lists, with one inner list per question sent. We sent one, so `chroma_hits` reads `[0]` of each.
- Chroma reports a **distance**, not a similarity. In the cosine space the distance is 1 minus the cosine similarity, so **lower is closer**.

**Predict first:** 2.3 gave kb-001 a cosine similarity of about 0.46. What number will Chroma show for it?

In [ ]:
result = collection.query(query_embeddings=[query_vector], n_results=3)
print(sorted(result.keys()))
print(chroma_hits(result))
print("1 - similarity from 2.3:", np.round(1 - scores[top], 2))

- kb-001 at distance 0.54, then kb-004 at 0.67 and kb-005 at 0.68: the same three as 2.3, and each distance is 1 minus its 2.3 similarity (0.54 = 1 - 0.46).
- The keys are `data`, `distances`, `documents`, `embeddings`, `ids`, `included`, `metadatas` and `uris`. By default the documents, metadatas and distances are filled in.
- Here the index returned the same three as the full comparison in 2.3. On 30 records that is expected; on millions an approximate index may miss a few.
- **Common mistake:** reading the distance as a score. Sorting Chroma's distances from high to low puts the worst match first, and the 0.68 article is the third-best, not the best.

**In an interview:** a Chroma query takes the question's vector and `n_results` and returns the nearest records with their distances. In the cosine space the distance is 1 minus the cosine similarity, so lower means closer.

### 6.3 Create, read, update, delete

Records were created by `add` in 6.1. The other three operations:

- `get(ids=[...])` reads records by id, with no ranking.
- `update(ids=..., documents=..., embeddings=...)` changes fields of existing records. The new text travels with its new vector, `embed([new_text])`. An id that is not found is logged and ignored.
- `delete(ids=[...])` removes records for good: there is no undo, and a missing id raises nothing.

The three cells below read kb-003, give it a new text, then delete kb-030, printing `count()` after each change.

**Predict first:** what will `count()` print after the update, and what after the delete?

In [ ]:
record = collection.get(ids=["kb-003"])
print(record["ids"], record["metadatas"][0]["title"])
print(record["documents"][0])

In [ ]:
new_text = ("Open Settings, then Profile, and enter the new email address. We send a six-digit code "
            "to the new address; type it on the Profile page within 15 minutes to finish the change.")
collection.update(
    ids=["kb-003"],
    documents=[new_text],
    embeddings=embed([new_text]),                     # the new vector travels with the new text
)
print(collection.get(ids=["kb-003"])["documents"][0])
print(f"count after update: {collection.count()}")

- The new text prints, and the count is still 30: an update changes a record and adds none.
- **Common mistake:** sending the new text without its vector. On this collection, with `embedding_function=None`, Chroma 1.5.9 refuses with `ValueError: You must provide an embedding function to compute embeddings.https://docs.trychroma.com/guides/embeddings in update.` (that is how the library prints it, with no space before the address). On a collection left on Chroma's default function, the same call would succeed quietly, with the text embedded by Chroma's own copy of the model instead of `embed`. Either way the fix is to pass the vector with every changed text.
- Try changing it: delete the line `embeddings=embed([new_text]),` and run the cell. You get the `ValueError` above, and kb-003 keeps the text it had. Put the line back and run the cell again: it succeeds and prints the new text.

In [ ]:
collection.delete(ids=["kb-030"])                     # gone for good: there is no undo
print(f"count after delete: {collection.count()}")
print("kb-030 now:", collection.get(ids=["kb-030"])["ids"])

- 29 after the delete, and `get` on kb-030 returns an empty list.
- **Common mistake:** trusting an `update` or a `delete` of an id that does not exist. Neither raises anything, so a typo in an id looks like success. `get` the id afterwards, or check the count.

**In an interview:** in Chroma, `add` creates records, `get` reads them by id and `query` by similarity, `update` changes them (a new text with its new vector), and `delete` removes them for good. Missing ids are skipped silently, so `count()` and `get` are the checks.

### 6.4 The data survives

A second `PersistentClient` on the same folder, then `get_collection("kb_articles", embedding_function=None)`. This cell does not remove the folder, because surviving data is its point.

**Predict first:** a brand-new client opens the same folder. How many records will it find?

In [ ]:
reopened = chromadb.PersistentClient(path="chroma_kb")      # no rmtree here: the data on disk is the point
kb_again = reopened.get_collection("kb_articles", embedding_function=None)
print(f"{kb_again.name}: {kb_again.count()} records")
print(kb_again.get(ids=["kb-003"])["documents"][0][:70])

- 29 records and kb-003's new text: the collection exactly as 6.3 left it, read from disk.
- It would survive a runtime restart, because `/content` survives one. It would not survive a new runtime, which starts with an empty `/content`; a real app keeps its folder on a disk that lasts, or runs a Chroma server.
- **Common mistake:** reopening without `embedding_function=None`. Chroma then attaches its default function to the reopened collection, and a later text sent without a vector is embedded by the wrong model.

**In an interview:** a `PersistentClient` stores its collections in a folder, so a new client on the same path finds them again with `get_collection`. An `EphemeralClient` keeps nothing after a restart.

**Before you move on**, check that you can:

- create a collection in the cosine space with no embedding function, and add records with ids, documents, embeddings and metadata;
- turn a Chroma distance into a cosine similarity, and say which way is closer;
- update a record's text with its new vector, and say what happens without the vector;
- delete a record and check the result with `count()` and `get`.

## 7. Pinecone overview

### 7.1 A managed vector database

**Pinecone** is a **managed vector database**: the index lives on Pinecone's servers, and your code reaches it over the internet with an API key. Its core features are the ones 4.1 named (an approximate nearest-neighbour index, metadata beside every vector, client libraries), run as a service, with no server for your team to look after.

- Four levels: an **organization** is your account and owns projects; a **project** holds indexes and the API keys that reach them; an **index** holds vectors, all of one length, compared with one metric; a **namespace** splits an index into parts. Namespaces are Wednesday's.
- A **record** is an id, its values (the vector) and optional metadata a query can filter on: a Chroma record without the separate document field.
- A **serverless index** keeps its data in object storage, a cheap cloud file store, and runs writes and reads on separate paths. Reads also check an in-memory table of recent writes, so new data is searchable right after it is written. The cost is a "cold" read: a namespace nobody has queried lately is fetched from object storage on its first query, which is slower. The older kind, pod-based indexes with hardware chosen in advance, is legacy; the notes page compares the two.
- In a retrieval workflow for a language model, Pinecone is the store the application searches for the passages it hands the model. Named only; week 9 wires it up.

The free **Starter** plan, as Pinecone's pricing page listed it when fetched on 2026-11-24:

| Limit | Starter plan |
|---|---|
| Storage | up to 2 GB |
| Read units | up to 1M a month |
| Write units | up to 2M a month |
| Egress | up to 1 GB a month |
| Indexes | up to 5, with 100 namespaces each |
| Projects and users | 1 project, up to 2 users |
| Region | AWS `us-east-1` only |

Usage past a monthly allowance is blocked rather than billed. A query costs 1 read unit per GB of namespace, at least 0.25; a fetch at least 1 read unit; an upsert, update or delete 1 write unit per KB, at least 5 per request.

**Predict first:** today stores 30 records and runs a few queries. How close does that come to the plan's monthly limits?

<details>
<summary>Check your answer</summary>

Nowhere near. 30 records of 384 numbers, about 1.5 KB each with their metadata, make one upsert of about 50 write units, against 2M a month. Ten queries cost about 2.5 read units, against 1M a month. This week keeps well inside the plan on purpose: at most two indexes at once, at most 1,000 vectors per index, and every index deleted at the end of its notebook.

</details>

- **Common mistake:** reading the limits as daily. The units are monthly, and usage past them is blocked, not billed.

**In an interview:** Pinecone is a managed vector database: indexes on Pinecone's servers, reached with an API key, holding records of id, values and metadata. Over a traditional database it gives similarity ranking, metadata filtering and scaling without running servers, and it is a common store for the retrieval step of retrieval-augmented generation.

### 7.2 Keys are secrets

An **API key** is a secret string that tells Pinecone which project a request belongs to. A key belongs to one project and is shown once, when it is created: it cannot be viewed again.

- **Making one.** In the Pinecone console, open the project, then API keys, then "+ API key". Name it, choose its permissions, press Create key, and copy it at once.
- **Never in a cell.** The demo repository and every learner repository are public, and bots scan GitHub for keys. A notebook is shared and pushed as a file, with its cells and outputs. Week 5 kept `ANTHROPIC_API_KEY` in an environment variable for the same reason.
- **Colab Secrets.** The panel behind the key icon in Colab's left bar. Add a secret named exactly `PINECONE_API_KEY`, paste the key as its value, and switch on **Notebook access**. Secrets belong to your Google account and are never saved in the notebook, so a saved copy holds no key.
- **The one line that reads it**, in 8.1: `pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))` after `from google.colab import userdata`. No variable holds the key and nothing prints it.
- **Two Colab errors, two fixes.** `SecretNotFoundError` ("Secret PINECONE_API_KEY does not exist.") means the name is wrong or missing: add the secret, spelled exactly. `NotebookAccessError` ("Notebook does not have access to secret PINECONE_API_KEY") means the secret exists but Notebook access is off: switch it on.
- **Permissions.** On a paid plan a key carries a role: ProjectEditor (read and write everything), ProjectViewer (read only), ControlPlaneEditor or Viewer (indexes and their settings), DataPlaneEditor or Viewer (records and namespaces). **Least privilege** means each program gets only what it needs: a read-only search service gets DataPlaneViewer. Starter keys can only have the All permission, so a free key can create and delete indexes, and needs guarding.
- **A leak.** Delete the key in the console at once; programs using it get a 401 error within seconds. Then create a new key and store it where the old one was. A planned **rotation** runs the other way round: create the replacement, switch to it, then delete the old key, so nothing breaks.

**Predict first:** a notebook with the key typed in a cell is pushed to a public repository. What is the first thing to do?

<details>
<summary>Check your answer</summary>

Delete the key in the Pinecone console straight away: that cuts off everything using it within seconds. Then make a new key and put it in Secrets. Deleting the cell or the commit is not enough, because the key may already have been copied.

</details>

- **Common mistake:** pasting the key into a cell "just to test", or printing `userdata.get(...)` to check it. The key is then in the notebook and its outputs. If it happens, delete the key in the console and make a new one; clearing the cell is not enough.

**In an interview:** keep an API key out of the code: in Colab Secrets in a notebook, in an environment variable elsewhere, read straight into the client. Give each program the least permission it needs, revoke a leaked key at once by deleting it, and rotate by creating the new key before deleting the old one.

**Before you move on**, check that you can:

- name Pinecone's four levels, from organization to namespace;
- quote two of the Starter plan's limits, and say what happens past one;
- say which error a misnamed secret gives and which one Notebook access switched off gives;
- list, in order, what to do when a key leaks.

## 8. Pinecone setup

### 8.1 Installing the client and connecting

- `%pip install -q pinecone==9.1.0`, the newest 9.x release. It adds two packages and changes none of Colab's own, so no restart.
- **Why not the newest (10.x).** Version 10 speaks a newer Pinecone API version, 2026-07, under which a new index can be a document index that refuses the vector writes taught today. 9.1.0 sends API version 2025-10, which has no document indexes, so the classic vector calls work: the ones every classic index and most existing code use.
- **The package name.** The client is `pinecone`; the old `pinecone-client` package is marked deprecated. Code with a top-level `pinecone.init(...)` is from before the rename, and on 9.1.0 that line raises `AttributeError`.
- **Connecting.** On Colab, `from google.colab import userdata`, then the one line from 7.2. The `else` branch is for running the notebook outside Colab, where `Pinecone()` reads the `PINECONE_API_KEY` environment variable by itself; it never runs on Colab.
- **The check.** `pc.list_indexes()` is the first call that reaches Pinecone. The cell prints how many indexes your project holds (0 on a fresh project) and whether `kb-tuesday` is one of them. A wrong key fails here with `UnauthorizedError: [401 ...]`, and the fix is to copy the key again or make a new one.

**Predict first:** your secret is named `PINECONE_KEY` instead of `PINECONE_API_KEY`. Which error will the connect cell raise?

<details>
<summary>Check your answer</summary>

`SecretNotFoundError`, because no secret has the exact name `PINECONE_API_KEY`. The fix is to rename the secret, not to make a new key.

</details>

In [ ]:
if IN_COLAB:
    %pip install -q pinecone==9.1.0
import pinecone
from pinecone import Pinecone, ServerlessSpec

print(f"pinecone {pinecone.__version__}")

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))   # from Secrets straight into the client
else:
    pc = Pinecone()   # outside Colab only: the client reads the PINECONE_API_KEY environment variable

names = [ix.name for ix in pc.list_indexes()]           # the first call that reaches Pinecone
print(f"connected: {len(names)} index(es) in this project; kb-tuesday among them: {'kb-tuesday' in names}")

- `pinecone 9.1.0`, then the connection line. A fresh project holds 0 indexes; ours held one unrelated to this notebook, and no `kb-tuesday`. If yours holds a `kb-tuesday` left by an earlier run, 8.2 skips the create and 9.5 removes it.
- The key went from Secrets into the client and appears nowhere in the notebook, in a cell or in an output.
- **Common mistake:** Notebook access left off. The secret exists, the cell raises `NotebookAccessError`, and it gets read as a wrong key, so a new key is made for nothing. The fix is the toggle in the Secrets panel.

**In an interview:** install the client with a pinned version, read the key from a secret store straight into `Pinecone(api_key=...)`, and make one cheap call such as `list_indexes()` to check the connection. A 401 means the key was received and rejected.

### 8.2 An index

- `if not pc.has_index("kb-tuesday"):` guards the create, so the cell can run twice without an error.
- `pc.create_index(name="kb-tuesday", dimension=384, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))`. `dimension=384` must match the vectors `embed` makes, `metric="cosine"` fixes how they are compared, and the spec places a serverless index in the free plan's only region.
- On 9.1.0 `create_index` waits, checking every 5 seconds, until the index is ready, so the next cell can write to it straight away.
- `pc.describe_index("kb-tuesday")` reads the index's settings and status, and `index = pc.Index("kb-tuesday")` returns the object that reads and writes records. Every call in section 9 goes through it.
- These `pc.*` calls are kept in 9.1.0 for compatibility; the newer spellings are `pc.indexes.create`, `.exists` and `.delete`. This week's notebooks keep the older ones.

**Predict first:** what will `status.ready` print straight after the create?

In [ ]:
start = time.perf_counter()
if not pc.has_index("kb-tuesday"):
    pc.create_index(
        name="kb-tuesday",
        dimension=384,                                        # must match the vectors embed makes
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"), # the free plan's only region
    )
print(f"create took {time.perf_counter() - start:.1f} s")

description = pc.describe_index("kb-tuesday")
print(description.name, description.dimension, description.metric, "ready:", description.status.ready)
index = pc.Index("kb-tuesday")

- `kb-tuesday 384 cosine ready: True`. On our run the create took about 5 seconds, the client checking every 5 seconds until the index was ready.
- The dimension and the metric are fixed when the index is made; index settings are Wednesday's.
- **Common mistake:** creating the index in a cell of your own without the `has_index` guard and running it twice. The second call fails because the index already exists. The guard is the habit.

**In an interview:** a Pinecone index is created once with a name, a dimension that matches the embedding model, a metric and a spec (serverless, in a cloud region). Guard the create with `has_index`, and reach the index through `pc.Index(name)`.

**Before you move on**, check that you can:

- write the pinned install line and the one line that reads the key from Secrets;
- create a serverless index of the right dimension behind a `has_index` guard;
- say why the dimension is 384;
- say why this week pins the 9.x client rather than 10.x.

## 9. CRUD with vector data

**CRUD** is short for create, read, update and delete, the four basic operations on stored records. Pinecone's index object has a method for each, and every one takes keyword arguments only: a positional `index.query(v, 3)` raises `TypeError: Index.query() takes 1 positional argument but 3 were given`.

### 9.1 Create

- One dictionary per article: `{"id": a["id"], "values": v.tolist(), "metadata": {...}}`, with the title, category, product and year as metadata. A Pinecone record has no document field, so the text stays in `KB`.
- Pinecone has no separate add: new records are written with `upsert`. What upsert does beyond adding is Wednesday's.
- `.tolist()` turns the vector into a plain Python list. This client never converts NumPy: an array in `values` raises `TypeError: Type is not JSON serializable: numpy.float32`, and a NumPy integer in metadata raises the same error for `numpy.int64`. The metadata comes from the `KB` dictionaries, which hold plain Python numbers.
- Pinecone is **eventually consistent**: a count or a read can lag a write by a few seconds. `wait_for_count(index, 30)` from the setup cell asks for the stats every 2 seconds, for up to 60 seconds, until the index holds 30 vectors.

**Predict first:** the cell reads the stats the instant the upsert returns, before waiting. What count might it show?

In [ ]:
records = [{"id": a["id"], "values": v.tolist(),                       # plain lists, never NumPy
            "metadata": {"title": a["title"], "category": a["category"],
                         "product": a["product"], "updated": a["updated"]}}
           for a, v in zip(KB, kb_vectors)]
index.upsert(vectors=records)
print(f"straight after the upsert: {index.describe_index_stats().total_vector_count} vectors")
wait_for_count(index, 30)

In [ ]:
stats = index.describe_index_stats()
print(f"total {stats.total_vector_count}, dimension {stats.dimension}")
print("per namespace:", {name: ns.vector_count for name, ns in stats.namespaces.items()})

- On our run the stats already showed 30 straight after the upsert, and `wait_for_count` printed `30 vectors in the index` at once. That is not guaranteed: the count can lag a write by a few seconds and show 0 or fewer than 30, which is why the next line waits instead of trusting the first number.
- `describe_index_stats()` returns the total, the dimension and a count per namespace. Records written without a namespace go to the default one, which this run listed under the key `__default__`.
- The upsert cost about 50 write units of the 2M a month.
- **Common mistake:** seeing 0 in a stats call of your own and upserting again. Nothing was lost; wait a few seconds, or poll as `wait_for_count` does.

**In an interview:** to create records in Pinecone, `upsert` a list of dictionaries with an id, the values as a plain list and metadata. Pinecone is eventually consistent, so a count read straight after a write can lag; poll the stats rather than writing again.

### 9.2 Read

- `index.query(vector=query_vector.tolist(), top_k=3, include_metadata=True)` ranks the records by similarity to the question's vector. Each match carries `id`, `score` and `metadata`, and `pinecone_hits` turns them into a table.
- Here the number is a **score**: the cosine similarity itself, so **higher is closer**, the opposite of Chroma's distance.
- `include_metadata=True` must be there: without it each match's `metadata` comes back as `None` and the table has no titles.
- `index.fetch(ids=["kb-003"])` reads records by id, with no ranking, and leaves out ids the index does not hold.

**Predict first:** Chroma gave kb-001 a distance of 0.54. What will Pinecone give it?

In [ ]:
result = index.query(vector=query_vector.tolist(), top_k=3, include_metadata=True)   # Try changing it: top_k=5
print(pinecone_hits(result))

In [ ]:
fetched = index.fetch(ids=["kb-003"])
print(fetched.vectors["kb-003"].metadata)

- kb-001 at 0.46, kb-004 at 0.33 and kb-005 at 0.32: the NumPy and Chroma top three, because all three stores hold the same vectors. Pinecone shows the similarity, so 0.46 here is Chroma's 0.54 read the other way.
- The fetch prints kb-003's metadata, with `updated` 2023. The update in 9.3 changes that year.
- **Common mistake:** calling `index.query(query_vector.tolist(), 3)` with positional arguments. It raises `TypeError: Index.query() takes 1 positional argument but 3 were given`: every call on the index is keyword-only.
- Try changing it: in the query cell, `top_k=5` and `vector=embed(["how long does delivery take"])[0].tolist()`, then rerun. kb-016 "Delivery times and costs" comes first at about 0.65, and four of the five are shipping articles. Put `vector=query_vector.tolist()` and `top_k=3` back and rerun the cell before 9.3.

**In an interview:** reading from Pinecone is either `query` (a vector and `top_k`, ranked by score, with `include_metadata=True` to get the fields back) or `fetch` (records by id, no ranking). With the cosine metric the score is the similarity, so higher is closer.

### 9.3 Update

- `index.update(id="kb-003", set_metadata={"updated": 2026})` sets or overwrites the named fields and leaves the rest alone.
- An update also applies after a short lag, so the cell fetches kb-003 again until the new year shows, checking once a second for up to 10 seconds.
- Chroma's `update` in 6.3 changed a record's text and its vector. Pinecone's `set_metadata` changes metadata only; writing a record's new values is Wednesday's upsert.

**Predict first:** we set only `updated`. What happens to kb-003's title, category and product?

In [ ]:
index.update(id="kb-003", set_metadata={"updated": 2026})   # changes this one field; the rest stay
for attempt in range(10):                                   # the change shows after a short lag
    metadata = index.fetch(ids=["kb-003"]).vectors["kb-003"].metadata
    if metadata["updated"] == 2026:
        break
    time.sleep(1)
print(metadata)

- `updated` is now 2026, and the title, category and product are unchanged: `set_metadata` merges into the metadata already there.
- **Common mistake:** a fetch straight after the update still shows 2023, and the update gets run again. Nothing failed; the change shows within seconds, which is why the cell fetches until it does.

**In an interview:** `update(id=..., set_metadata={...})` changes the named metadata fields of one record and keeps the rest. It applies after a short lag, so check with a `fetch` before relying on it.

### 9.4 Delete

- `index.delete(ids=["kb-030"])` removes the record. It cannot be undone, and ids the index does not hold are ignored.
- The count and a fetch can each lag the delete, so the cell waits for the count, then fetches kb-030 until it is gone, checking once a second for up to 10 seconds.
- Each delete request costs at least 5 write units, like every write.

**Predict first:** what will `wait_for_count` print after deleting kb-030?

In [ ]:
index.delete(ids=["kb-030"])                          # no undo; an id the index does not hold is ignored
wait_for_count(index, 29)
for attempt in range(10):                             # a fetch can lag the delete too
    gone = "kb-030" not in index.fetch(ids=["kb-030"]).vectors
    if gone:
        break
    time.sleep(1)
print("kb-030 gone from fetch:", gone)

- `29 vectors in the index`, then `kb-030 gone from fetch: True`. On our first run, a single fetch made the moment the count reached 29 still returned kb-030: the count and the fetch caught up at different moments, which is why the cell checks both.
- **Common mistake:** deleting an id with a typo. Pinecone ignores ids it does not hold and raises nothing, so the count stays at 30. The count is the check.

**In an interview:** `delete(ids=[...])` removes records for good, in Pinecone as in Chroma. Unknown ids are ignored, so check the count afterwards.

### 9.5 Clean up and compare

The last Pinecone cell deletes the index, behind the same `has_index` guard the create used, so a second run raises nothing. It keeps your free project empty between days, and the next run starts clean.

**Predict first:** after the clean-up, will `kb-tuesday` still be in the list of indexes?

In [ ]:
if pc.has_index("kb-tuesday"):
    pc.delete_index("kb-tuesday")
print("kb-tuesday still there:", "kb-tuesday" in [ix.name for ix in pc.list_indexes()])

- `kb-tuesday still there: False`. Wednesday's notebook creates its own indexes.
- The three stores, operation by operation:

| Operation | SQLite | Chroma | Pinecone |
|---|---|---|---|
| Create | `INSERT` | `add(...)` | `upsert(vectors=...)` |
| Read by id | `SELECT ... WHERE id = ?` | `get(ids=...)` | `fetch(ids=...)` |
| Read nearest | read every row and compare | `query(query_embeddings=..., n_results=3)` | `query(vector=..., top_k=3, include_metadata=True)` |
| Update | `UPDATE ... SET` | `update(ids=..., documents=..., embeddings=...)` | `update(id=..., set_metadata=...)` |
| Delete | `DELETE ... WHERE` | `delete(ids=...)` | `delete(ids=...)` |

- SQLite is the one with no index for nearest-neighbour search: it reads every row and compares.
- One hidden difference: Chroma's `add` skips an id that already exists, with no error. What Pinecone's `upsert` does with one is Wednesday's.
- Chroma suits a prototype or a small app on one machine. Pinecone suits data that must be shared and served without running a server.
- **Common mistake:** skipping the clean-up. The index stays in your project overnight, and this week's notebooks keep to at most two indexes at once.

**In an interview:** create, read, update and delete line up across the three stores: insert or add or upsert; select, get or fetch by id and query by similarity; update; delete. Only the vector databases find the nearest vectors without reading every row, and a Pinecone index is deleted when the work is done to stay inside the free plan.

**Before you move on**, check that you can:

- upsert records with plain-list vectors and metadata, and wait for the count;
- query an index and fetch a record by id, and read a Pinecone score against a Chroma distance;
- change one metadata field with `update`, and check it with `fetch`;
- delete a record and an index, each behind the right check.

## Recap

| Idea | What it is for | Watch out for |
|---|---|---|
| Vectors and cosine similarity | placing things as lists of numbers, and saying how alike two are by direction | cosine ignores length, so a doubled vector scores 1.0; a learned embedding's numbers have no names |
| Similarity search with `embed` | finding stored texts by meaning, nearest first | the same model for texts and questions; `np.argsort(-scores)`, not `np.argsort(scores)`; read the order and the gap, not a fixed cut-off |
| Vectors in SQLite | storing a vector as bytes beside ordinary columns | `dtype=np.float32` when reading back; `?` placeholders; search time grows with every row |
| Chroma | a vector database inside the notebook, on disk with `PersistentClient` | distance, lower is closer; with `embedding_function=None` every changed text needs its vector; `add` skips an existing id silently |
| Pinecone setup | a managed index reached with a key kept in Secrets | the key never goes in a cell; `SecretNotFoundError` and `NotebookAccessError` have different fixes; pin 9.1.0 |
| Pinecone CRUD | upsert, query and fetch, update, delete on a hosted index | `.tolist()` on every vector; keyword arguments only; `include_metadata=True`; counts lag a write; delete the index at the end |

**Quick check**
1. What is a vector embedding, and what is the cosine similarity of `[3, 4]` and `[6, 8]`?
2. Walk me through turning a text into a vector and searching with it.
3. Why not keep vectors in an ordinary SQL table at scale?
4. When would you use `EphemeralClient` or `PersistentClient` in Chroma, and what does a distance of 0.54 mean?
5. How do you give a notebook a Pinecone key without writing it in the code, and what do you do if one leaks?
6. Show create, read, update and delete on a Pinecone index. Why can the stats show 0 right after an upsert?

<details>
<summary>Answers (say yours aloud first)</summary>

1. A list of numbers that represents a piece of data as a point, placed so that alike things sit close together. The cosine similarity is 50 / (5 × 10) = 1.0: the second vector is the first doubled, and cosine ignores length.
2. Embed every stored text with one model, here 384 numbers of length 1 each. Embed the question with the same model, score it against every stored vector with a dot product, which equals the cosine similarity for vectors of length 1, and return the top few, nearest first.
3. A table can store vectors as bytes, but it has no vector index, so every search reads and compares every row, and doubling the rows roughly doubles the time. A vector database's approximate nearest-neighbour index, such as HNSW, finds near vectors without the full scan.
4. `EphemeralClient` for a throwaway test, since its data is gone on restart; `PersistentClient` to keep the data in a folder on disk. 0.54 is a cosine distance, 1 minus the similarity, so lower is closer and it means a similarity of about 0.46.
5. Keep it in Colab Secrets as `PINECONE_API_KEY` with Notebook access on, and read it with `userdata.get` straight into `Pinecone(api_key=...)`, so no cell or output holds it. If it leaks, delete it in the Pinecone console at once, then create a new key and update the secret.
6. `upsert` records with an id, the values as a plain list and metadata; `query` with a vector and `top_k`, or `fetch` by id; `update` with `set_metadata`; `delete` with a list of ids. Pinecone is eventually consistent, so the count catches up within seconds: poll the stats instead of writing again.

</details>